In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [4]:
v1.dot(dv)

np.float32(0.32332397)

In [5]:
q2 = "How to installl DOcker on Windows?"
v2 = model.encode(q2)

In [6]:
v2.dot(dv)

np.float32(0.020419423)

In [7]:
from ingest import load_faq_data

documents = load_faq_data()

In [8]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [9]:
from tqdm.auto import tqdm

In [10]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)
    
len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [11]:
import numpy as np
X = np.array(vectors)

In [12]:
X.shape

(1350, 384)

In [13]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [14]:
scores = X.dot(v_query)

In [15]:
scores

array([ 0.48740575,  0.20991933,  0.762941  , ..., -0.08637968,
        0.03759793, -0.03037044], shape=(1350,), dtype=float32)

In [16]:
scores = [v_query.dot(X[i]) for i in range(len(X))]
print(scores)

[np.float32(0.48740578), np.float32(0.20991932), np.float32(0.76294106), np.float32(0.4437854), np.float32(0.2608399), np.float32(0.48665166), np.float32(0.30061564), np.float32(0.56009996), np.float32(0.4596049), np.float32(0.2562817), np.float32(0.33153275), np.float32(0.27318534), np.float32(0.27691638), np.float32(0.34123003), np.float32(0.46007168), np.float32(0.26240283), np.float32(0.3920008), np.float32(0.10854167), np.float32(0.2756731), np.float32(0.16646823), np.float32(0.31437925), np.float32(0.006685449), np.float32(0.112050325), np.float32(0.21905689), np.float32(0.3400085), np.float32(0.23571292), np.float32(0.32714835), np.float32(0.15088356), np.float32(0.16563278), np.float32(0.05545026), np.float32(0.23541194), np.float32(0.08533022), np.float32(-0.0030899365), np.float32(-0.042598385), np.float32(-0.06027717), np.float32(0.006491054), np.float32(0.034277476), np.float32(-0.0495897), np.float32(-0.0006708354), np.float32(-0.017013595), np.float32(0.070627116), np.flo

In [17]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [18]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [19]:
top5 = np.argsort(scores)[-5:]

In [23]:
top5 = top5[::-1]
top5

array([  7, 538, 907, 625,   2])

In [24]:
top5 = np.argsort(-scores)[:5]

TypeError: bad operand type for unary -: 'list'

In [25]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.56009996
{'id': '068529125b', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}

0.6536312
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date

In [26]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [27]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [28]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [29]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [30]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [31]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [32]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)

In [33]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client = openai_client
)

In [34]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [35]:
class RAGVector(RAGBase):
    
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder
        
    def search (self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}
        
        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict = filter_dict
        )

In [36]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client
)

In [37]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes. You can still join, but if you want a certificate, you need to submit your project while submissions are still open.'

In [45]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vector2.db"
)

In [46]:
vs_index.fit(vectors, documents)

In [47]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [48]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [49]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [50]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [51]:
vs_index.close()